# Tiny I-JEPA on CIFAR-10: masking geometry and robustness

This notebook is the assignment's Colab notebook. Run it top to bottom in a fresh runtime
(**Runtime → Run all**). A GPU runtime is recommended. On a CPU runtime it falls back to a short smoke run.

| § | Content |
|---|---|
| 1 | Setup: upstream repository (attribution, pinned commit), this repository, dependencies, settings |
| 2 | Code: what changed relative to upstream `ijepa.py` (diff) and the research modification |
| 3 | CPU smoke test: one batch through the model, loss, backward pass, one optimiser step, EMA update |
| 4 | Automated tests (`pytest`) |
| 5 | Data splits: creation, sizes, disjointness, recorded indices and seeds |
| 6 | Stress tests: occlusion 10/25/40 % and Gaussian noise σ = 0.05/0.15, with fixed seeds |
| 7 | Masks: baseline, proposed and ablation |
| 8 | Pretraining: baseline, proposed, ablation (same data, seeds and budget) |
| 9 | Linear-probe evaluation: frozen encoder, random-init control, all stress tests |
| 10 | Results: tables, loss curves, robustness plots, training graphs |
| 11 | Findings, including the report's 15k-step results |

The code lives in the repository (`models/`, `data/`, `train.py`, `evaluate.py`, `scripts/`), so the notebook only drives it.
Training and evaluation use `scripts/train.sh` and `scripts/evaluate.sh`, the same commands as on any other machine.

**Budget.** `STEPS = 500` is the assignment's initial budget (about 1–2 min per run on a Colab T4, plus evaluation).
`STEPS = 15000` reproduces the runs of the report (about 10 min per run on an L40S, more on a T4).

## 1. Setup

The first code cell clones this repository (`REPO_URL`). If the notebook is opened from a checkout of the repository
(`notebooks/` or the repository root), no clone is needed.

The upstream repository [keon/jepa](https://github.com/keon/jepa) (MIT) is cloned at the commit used for every run.
It is used for attribution, for the diff, and for checking that the model classes are identical.
Its `requirements.txt` is **not** installed, because it pins `torch==2.11.0` and Colab would then reinstall PyTorch.
The notebook uses Colab's preinstalled torch, torchvision, numpy, pandas and matplotlib (versions are printed below),
and installs only `pyyaml` and `pytest`.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/alfahimmohammad/tiny-jepa-vision-research.git"   # this repository
UPSTREAM_URL = "https://github.com/keon/jepa.git"
UPSTREAM_COMMIT = "d55bab7b712d4dfa8bcfb1afc9196e454a56dde7"
IN_COLAB = "google.colab" in sys.modules


def sh(cmd, env=None):
    # Run a shell command, stream its output, fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


def find_repo():
    for d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/tiny-jepa-vision-research"):
        if os.path.exists(os.path.join(d, "models", "modified_ijepa.py")):
            return os.path.abspath(d)


REPO = find_repo()
if REPO is None:
    sh(f"git clone -q {REPO_URL} tiny-jepa-vision-research")
    REPO = os.path.abspath("tiny-jepa-vision-research")
UPSTREAM = os.environ.get("JEPA_UPSTREAM", os.path.join(os.path.dirname(REPO), "jepa"))
if not os.path.exists(os.path.join(UPSTREAM, "ijepa.py")):
    sh(f"git clone -q {UPSTREAM_URL} {UPSTREAM}")
if os.path.exists(os.path.join(UPSTREAM, ".git")):
    sh(f"git -C {UPSTREAM} checkout -q {UPSTREAM_COMMIT} && git -C {UPSTREAM} log -1 --format='upstream commit %H'")
os.chdir(REPO)
sys.path.insert(0, REPO)
if IN_COLAB:
    sh("pip -q install pyyaml pytest")
print("repository:", REPO, "\nupstream:  ", UPSTREAM)

In [ ]:
import json, platform, time, random, copy
import numpy as np, pandas as pd, torch, torchvision, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HARDWARE = torch.cuda.get_device_name() if DEVICE == "cuda" else platform.processor() or "CPU"

# ---- experiment settings (every run in this notebook uses them) ----
STEPS = 500                       # optimizer steps per run (assignment budget); 15000 = the report's budget
SEEDS = [0]                       # pretraining seeds; e.g. [0, 1, 2] for mean +- std
VARIANTS = {"baseline": "configs/baseline.yaml",    # original Tiny I-JEPA (upstream masks, model, loss, EMA)
            "proposed": "configs/proposed.yaml",    # one large contiguous target block, 20 context / 32 target tokens
            "ablation": "configs/ablation.yaml"}    # same token counts, targets in many small scattered blocks
DATA_ROOT = os.environ.get("DATA_ROOT", "./cifar_data")
RUNS = os.path.abspath("runs_notebook")             # run outputs of this notebook
METRICS_CSV = os.path.join(RUNS, "metrics.csv")      # (results/metrics.csv holds the report runs)
MIXED_PROBE = True                                   # diagnostic probe trained on clean + noisy images

if DEVICE == "cpu":                                   # no GPU: demonstrate the pipeline on a short run only
    STEPS, MIXED_PROBE = min(STEPS, 20), False
    print("No GPU: STEPS reduced to", STEPS, "and the mixed probe is skipped (CPU smoke run, not a result).")

ENV = {"DATA_ROOT": DATA_ROOT, "RUNS": RUNS, "METRICS_CSV": METRICS_CSV, "PYTHON": sys.executable}
os.makedirs(RUNS, exist_ok=True)
print(f"python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"numpy {np.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"device: {DEVICE} ({HARDWARE}) | steps {STEPS} | seeds {SEEDS} | data {DATA_ROOT}")

## 2. Code: what changed relative to upstream

`models/modified_ijepa.py` is a copy of upstream `ijepa.py`. The classes `Encoder` and `Predictor`, the functions
`sample_ijepa_masks`, `ema_update`, `lr_warmup_cosine` and `param_groups`, and the loss formula are byte-identical.
Only `train()` changed:

1. **Step-limited training.** A fixed number of optimizer steps over the 10k pretraining split replaces the epoch loop.
   The warmup-cosine LR and the EMA schedule (0.996 → 1) span the step budget.
2. **Pluggable masks.** A `mask_fn(B, grid, rng)` from `models/masks.py` replaces the call to `sample_ijepa_masks`.
   The `upstream` strategy gives exactly the original masks, and this is tested.
3. **Context noise (H3, optional).** Noise on the context-encoder input only. It is off in all three runs here.

The loss is computed per target block and averaged over blocks, exactly as upstream.
Target/context separation, stop-gradient on the target branch, and EMA-only target updates are unchanged.
A1 (the code-level explanation) is in Sec. 2 of the report and in `docs/A1_code_explanation.md`.

In [ ]:
# Full diff against upstream: only train() and the imports/docstring differ.
sh(f"diff {UPSTREAM}/ijepa.py models/modified_ijepa.py || true")

**The research modification (Part B): masking geometry, hypothesis H1.** The proposed sampler `1_large` hides **one
large contiguous target block**, of 35–42 patches. Per image it then draws exactly 20 visible context tokens and
32 target tokens.

The ablation `scattered` keeps those counts, 20 context and 32 target tokens, but spreads the targets over many small
non-overlapping blocks of 2–6 patches. This removes the contiguity of the hidden region.

Comparing the two therefore isolates geometry from token counts. Sec. 4 and Table 2 of the report describe every
sampler in detail (block scales, aspect ratios, context block).

In [ ]:
for f in ("configs/baseline.yaml", "configs/proposed.yaml", "configs/masks/1_large.yaml",
          "configs/ablation.yaml", "configs/masks/scattered.yaml"):
    print(f"# ---- {f}\n" + open(f).read())

## 3. CPU smoke test

This is the assignment's minimal model, built on the **CPU**. It runs a batch of 8 random images through the model,
computes the smooth-L1 loss over the 4 target blocks, runs the backward pass and one AdamW step, then applies the EMA
update. The checks:

- shapes are correct, and outputs, loss and gradients are finite;
- the target encoder receives no gradient and changes only through the EMA update;
- the upstream model classes give exactly the same outputs as the repository's copy.

A 2-step run of the repository's `train()` on CPU follows. That is the function `train.py` uses.

In [ ]:
import importlib.util
from models.modified_ijepa import Encoder, Predictor, sample_ijepa_masks, ema_update, train
import torch.nn.functional as F

spec = importlib.util.spec_from_file_location("ijepa_upstream", os.path.join(UPSTREAM, "ijepa.py"))
up = importlib.util.module_from_spec(spec); spec.loader.exec_module(up)

torch.manual_seed(0)
ctx_enc = Encoder(img_size=32, patch_size=4, dim=128, depth=6, heads=4)
tgt_enc = copy.deepcopy(ctx_enc)
for p in tgt_enc.parameters():
    p.requires_grad_(False)
pred = Predictor(grid=ctx_enc.grid, enc_dim=128, dim=64, depth=4, heads=4)
opt = torch.optim.AdamW(list(ctx_enc.parameters()) + list(pred.parameters()), lr=3e-4)
print("trainable parameters:", sum(p.numel() for m in (ctx_enc, pred) for p in m.parameters()))

imgs = torch.randn(8, 3, 32, 32)
ctx, tgts = sample_ijepa_masks(8, ctx_enc.grid, rng=random.Random(0))            # 1. context / target indices
ci = torch.tensor(ctx)
assert all(set(c).isdisjoint(t[b]) for t in tgts for b, c in enumerate(ctx)), "target leaked into context"
with torch.no_grad():                                                             # 2. targets without gradients
    full = F.layer_norm(tgt_enc(imgs), (128,))
ce = ctx_enc(imgs, ci)                                                            # context encoder: visible patches only
loss = sum(F.smooth_l1_loss(pred(ce, ci, torch.tensor(t)),                       # 3.-4. predict, smooth-L1
                            full.gather(1, torch.tensor(t).unsqueeze(-1).expand(-1, -1, 128))) for t in tgts) / len(tgts)
opt.zero_grad(); loss.backward()
grads = [p.grad for m in (ctx_enc, pred) for p in m.parameters()]
assert torch.isfinite(loss) and all(g is not None and torch.isfinite(g).all() for g in grads)
assert all(p.grad is None for p in tgt_enc.parameters()), "target encoder received a gradient"
before = copy.deepcopy(tgt_enc.state_dict())
opt.step()                                                                        # 5. update context encoder + predictor
assert all(torch.equal(before[k], v) for k, v in tgt_enc.state_dict().items()), "optimizer touched the target encoder"
ema_update(tgt_enc, ctx_enc, 0.996)                                               #    then the EMA target
for (k, v), p in zip(tgt_enc.state_dict().items(), ctx_enc.state_dict().values()):
    assert torch.allclose(v, 0.996 * before[k] + 0.004 * p, atol=1e-6)
print(f"shapes: context tokens {tuple(ce.shape)}, target features {tuple(full.shape)}, "
      f"{len(tgts)} target blocks of {[len(t[0]) for t in tgts]} tokens | loss {loss.item():.4f} | checks passed")

# the upstream classes give the same outputs with the same weights
u_enc = up.Encoder(img_size=32, patch_size=4, dim=128, depth=6, heads=4); u_enc.load_state_dict(ctx_enc.state_dict())
assert torch.equal(u_enc(imgs, ci), ctx_enc(imgs, ci)), "differs from upstream Encoder"
print("upstream Encoder == repository Encoder (same weights, same output)")

# the repository's training function, 2 steps on CPU
ds = [torch.randn(3, 32, 32, generator=torch.Generator().manual_seed(i)) for i in range(16)]
out = train(ds, steps=2, batch_size=8, num_workers=0, device="cpu")
print("train() on CPU:", [round(r["loss"], 4) for r in out["log"]])

## 4. Automated tests

49 pytest tests run on the CPU in about 15 s. They cover:

- **Masks:** valid indices, context/target disjoint, the declared token counts and the intended geometry for every
  mask strategy.
- **Shapes:** correct shapes and finite outputs.
- **Target encoder:** it receives no gradient and changes only through the EMA update.
- **Context encoding:** it does not depend on target pixels.
- **Probe training:** it leaves the frozen encoder's parameters unchanged.
- **Splits and corruptions:** splits are disjoint and deterministic, and corruption seeds are matched.
- **Configs:** a 2-step smoke run of every experiment config.

In [ ]:
sh(f"{sys.executable} -m pytest -q")

## 5. Data splits

The splits are created from `torchvision.datasets.CIFAR10` with fixed seeds (`data/splits.py`).

- **Pretraining (10,000):** a uniform random subset of the official training split. Labels are not used to choose it.
- **Probe-train (2,000) and validation (1,000):** class-stratified subsets taken from the remaining training images.
- **Test (2,000):** a class-stratified subset of the official test split.

The indices and seeds are saved in `results/splits.json`. The splits are regenerated here and must reproduce its
fingerprint, so every run uses exactly the same images.

In [ ]:
from data.splits import load_cifar, make_splits, load_or_create_splits, to_unit_tensor, SPLIT_SEED, TEST_SEED
xtr, ytr, xte, yte = load_cifar(DATA_ROOT)
fresh = make_splits(ytr, yte)
saved = load_or_create_splits("results/splits.json", ytr, yte)       # raises if they differ
ind = saved["indices"]
assert fresh["fingerprint"] == saved["fingerprint"] == "f8c34b6bbf7f644f"
for a in ("pretrain", "probe_train", "val"):
    for b in ("pretrain", "probe_train", "val"):
        assert a == b or not set(ind[a]) & set(ind[b]), f"{a} and {b} overlap"
lab = lambda k: ytr if saved["source"][k] == "train" else yte
rows = []
for k, v in ind.items():
    bc = np.bincount(lab(k)[v], minlength=10)
    rows.append({"split": k, "source": f"official {saved['source'][k]}", "images": len(v),
                 "per class (min-max)": f"{bc.min()}-{bc.max()}", "first indices": v[:5]})
display(pd.DataFrame(rows).set_index("split"))
print(f"split seed {SPLIT_SEED} (official train split), test seed {TEST_SEED} (official test split), "
      f"fingerprint {saved['fingerprint']}; all indices: results/splits.json")

## 6. Stress tests

All corruptions are applied to the same 2,000 test images in [0, 1] pixel space, **before** CIFAR-10 normalisation
(`data/corruptions.py`). Each one has its own fixed seed, so every model sees byte-identical inputs and the
comparisons between models are paired.

- **Occlusion.** One axis-aligned square per image, with side round(√(f·32²)) px: 10, 16 and 20 px, covering 9.8,
  25.0 and 39.1 % of the area. The top-left corner is uniform over all positions that keep the square inside the
  image. The fill value is the CIFAR-10 per-channel mean, which becomes exactly 0 after normalisation.
- **Gaussian noise.** x + N(0, σ²) per pixel and channel, with σ ∈ {0.05, 0.15}, then clipped to [0, 1].

The linear probe is trained on clean images only and applied unchanged to every condition.

In [ ]:
from data.corruptions import build_conditions
from models.modified_ijepa import MEAN, STD
test01 = to_unit_tensor(xte, ind["test"])
conds, cond_meta = build_conditions(test01, MEAN)
display(pd.DataFrame(cond_meta).T)
fig, ax = plt.subplots(3, 6, figsize=(10, 5.4))
for r, i in enumerate([0, 1, 2]):
    for c, (name, x) in enumerate(conds.items()):
        ax[r, c].imshow(x[i].permute(1, 2, 0).numpy()); ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(name, fontsize=9)
plt.suptitle("The same test images under every stress test (fixed seeds)", fontsize=10); plt.tight_layout(); plt.show()

## 7. Masks

The examples below show one batch from each sampler. Grey cells are visible context patches; each coloured cell
belongs to one target block, the set its loss term is computed on; white cells are neither.

- **baseline:** 4 target blocks of the upstream size, about 11 tokens each. The context is trimmed to the batch
  minimum, about 20 tokens.
- **proposed:** 1 large block, 20 context and 32 target tokens.
- **ablation:** 8–16 small blocks, 20 context and 32–36 target tokens.

In [ ]:
from models.masks import mask_sampler
from train import load_config
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#ffffff", "#c9c8c4"] + ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6", "#e34948",
                                                 "#008300", "#e87ba4"] * 3)
fig, ax = plt.subplots(len(VARIANTS), 6, figsize=(9, 1.6 * len(VARIANTS)))
for r, (v, cfg_path) in enumerate(VARIANTS.items()):
    ctx, tgts = mask_sampler(load_config(cfg_path)["mask"])(64, 8, random.Random(1))
    for c in range(6):
        g = np.zeros(64)
        g[ctx[c]] = 1
        for k, t in enumerate(tgts):
            g[t[c]] = 2 + k
        ax[r, c].imshow(g.reshape(8, 8), cmap=cmap, vmin=0, vmax=cmap.N - 1)
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
    ax[r, 0].set_ylabel(v, fontsize=9)
    print(f"{v:9s}: context {len(ctx[0])} tokens | {len(tgts)} target sets of {[len(t[0]) for t in tgts]} tokens")
plt.tight_layout(); plt.show()

## 8. Pretraining

Each variant is trained with `scripts/train.sh <config> <seed> <steps>`. All runs use identical data, data order and
seeds:

- the 10k unlabelled pretraining images; labels never reach the encoder;
- batch size 64, AdamW, lr 3·10⁻⁴ with 5 % warmup and cosine decay, wd 0.05, FP32;
- EMA momentum 0.996 → 1.

Every 50 steps a fixed-setting probe is evaluated. This only produces the curves and is never used for selection.

**Checkpoint rule, fixed in advance:** the final step is the checkpoint that gets evaluated.

In [ ]:
RUN_DIRS = {}
for v, cfg_path in VARIANTS.items():
    for seed in SEEDS:
        name = load_config(cfg_path)["name"]
        run_dir = os.path.join(RUNS, f"{name}_s{seed}_{STEPS}steps")
        if not os.path.exists(os.path.join(run_dir, "summary.json")):
            sh(f"bash scripts/train.sh {cfg_path} {seed} {STEPS}", env=ENV)
        RUN_DIRS[(v, seed)] = run_dir
print(json.dumps({f"{v}/s{s}": d for (v, s), d in RUN_DIRS.items()}, indent=1))

## 9. Linear-probe evaluation

Each run is evaluated with `scripts/evaluate.sh <run_dir>` (`evaluate.py`):

- **Features.** The frozen context encoder sees all 64 patches, with no patch mask. Its tokens are mean-pooled to 128
  dimensions and standardised with probe-train statistics.
- **Probe.** A linear layer trained for 100 epochs with AdamW. (lr, wd) is selected on the 1k validation images only,
  and the test set is never used for selection.
- **Testing.** The clean-trained probe is applied to all six test conditions.
- **Same protocol for every encoder,** including the **randomly initialised** encoder of each run (the step-0
  weights), which is the control.
- **Also reported:** the held-out latent loss on the fixed validation images and fixed masks, collapse statistics,
  and an oracle probe, which is a diagnostic only.

An assertion checks that the encoder's parameter hash is unchanged by probing.

In [ ]:
for (v, seed), run_dir in RUN_DIRS.items():
    if not os.path.exists(os.path.join(run_dir, "eval.json")):
        sh(f"bash scripts/evaluate.sh {run_dir}" + ("" if MIXED_PROBE else " --no_mixed_probe"), env=ENV)

# paired comparison: identical splits and corruption inputs in every evaluation
evals = {k: json.load(open(os.path.join(d, "eval.json"))) for k, d in RUN_DIRS.items()}
assert len({e["splits_fingerprint"] for e in evals.values()}) == 1
assert len({json.dumps({c: m.get("sha256") for c, m in e["corruptions"].items()}, sort_keys=True)
            for e in evals.values()}) == 1
print("all runs: splits", next(iter(evals.values()))["splits_fingerprint"], "and identical corruption tensors (sha256)")

## 10. Results

In [ ]:
CONDS = ["clean", "occ10", "occ25", "occ40", "noise0.05", "noise0.15"]
rows = []
for (v, seed), e in evals.items():
    for r in e["rows"]:
        rows.append({"variant": "random init (control)" if r["encoder"] == "random_init" else v, "seed": seed, **r})
R = pd.DataFrame(rows)
R = R[~((R.variant == "random init (control)") & R.duplicated(["variant", "seed", "acc_clean"]))]   # one control per seed
order = ["random init (control)"] + list(VARIANTS)

def table(cols, scale=1.0, digits=1):
    g = R.groupby("variant", sort=False)[cols]
    m, s = (g.mean() * scale).reindex(order), (g.std() * scale).reindex(order)
    if len(SEEDS) == 1:
        return m.round(digits)
    return m.round(digits).astype(str) + " ± " + s.round(digits).astype(str)

display(Markdown(f"**Test accuracy (%), validation-selected probe** ({STEPS} steps, seeds {SEEDS}, {HARDWARE})"))
display(table([f"acc_{c}" for c in CONDS], 100).rename(columns=lambda c: c[4:]))
display(Markdown("**Held-out latent loss and collapse diagnostics** (loss: fixed validation images and masks; "
                 "other ctx = another image's context; content-free = per-position mean target)"))
display(table(["heldout_loss", "heldout_loss_other_ctx", "heldout_loss_mean_target", "feat_std_mean", "eff_rank",
               "top1_sv_frac", "mean_pairwise_cos"], 1, 3).rename(columns={
    "heldout_loss": "held-out loss", "heldout_loss_other_ctx": "loss, other ctx", "heldout_loss_mean_target":
    "loss, content-free", "feat_std_mean": "feature std", "eff_rank": "eff. rank (/128)", "top1_sv_frac":
    "top-1 SV share", "mean_pairwise_cos": "pairwise cos"}))
cost = R[R.variant != "random init (control)"].groupby("variant", sort=False)[
    ["params_trainable", "steps", "train_time_s", "peak_mem_mb"]].mean().round(1)
cost["hardware"] = HARDWARE
display(Markdown("**Parameters, steps, runtime and memory**")); display(cost)

In [ ]:
# Pretraining loss and held-out latent loss of the three variants (seed SEEDS[0])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
COL = {"baseline": "#8a8986", "proposed": "#2a78d6", "ablation": "#eb6834"}
for v in VARIANTS:
    d = RUN_DIRS[(v, SEEDS[0])]
    L, E = pd.read_csv(f"{d}/train_log.csv"), pd.read_csv(f"{d}/eval_curve.csv")
    w = max(1, STEPS // 50)
    ax[0].plot(L.step, L.loss.rolling(w, min_periods=1).mean(), color=COL[v], label=v)
    ax[1].plot(E.step, E.heldout_loss, "o-", ms=3, color=COL[v], label=v)
    ax[2].plot(E.step, 100 * E.val_acc, "o-", ms=3, color=COL[v], label=v)
for a, t in zip(ax, [f"training loss ({w}-step mean)", "held-out latent loss (val, fixed masks)",
                     "val accuracy (%), fixed probe setting"]):
    a.set_title(t, fontsize=10); a.set_xlabel("pretraining step"); a.grid(alpha=0.3)
ax[0].legend(); plt.tight_layout(); plt.show()

# Robustness: test accuracy per condition, clean-trained probe
acc = table([f"acc_{c}" for c in CONDS], 100) if len(SEEDS) == 1 else \
      R.groupby("variant", sort=False)[[f"acc_{c}" for c in CONDS]].mean().reindex(order) * 100
fig, ax = plt.subplots(figsize=(10, 3.4))
x, w = np.arange(len(CONDS)), 0.8 / len(order)
for k, v in enumerate(order):
    ax.bar(x + (k - (len(order) - 1) / 2) * w, acc.loc[v].values, w * 0.92, label=v,
           color={"random init (control)": "#d6d5d0", **COL}[v])
ax.set_xticks(x); ax.set_xticklabels(["clean", "occlusion 10%", "occlusion 25%", "occlusion 40%", "noise σ=0.05",
                                      "noise σ=0.15"])
ax.set_ylabel("test accuracy (%)"); ax.axhline(10, color="#888", ls=":", lw=1)
ax.set_ylim(0, 1.2 * np.nanmax(acc.values.astype(float))); ax.legend(ncol=4, fontsize=8, frameon=False, loc="upper center")
ax.set_title(f"Robustness, frozen encoder + linear probe trained on clean images ({STEPS} steps)", fontsize=10)
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

**Training graphs.** These come from `scripts/plot_run.py`, the same plots the development notebooks printed during
training. Every panel shows all three runs:

- the training signals;
- the periodic evaluation;
- the test accuracy per stress test across training.

In [ ]:
spec = importlib.util.spec_from_file_location("plot_run", "scripts/plot_run.py")
pr = importlib.util.module_from_spec(spec); spec.loader.exec_module(pr)
pr.OUT = RUNS
names = [pr.load_run(RUN_DIRS[(v, SEEDS[0])]) for v in VARIANTS]
pr.plot_training(names); pr.plot_metrics(names); pr.plot_robustness_curves(names)
for f in ("training.png", "metrics.png", "robustness.png"):
    display(Image(os.path.join(RUNS, f)))

## 11. Findings

### 11a. This notebook's runs

In [ ]:
m = R.groupby("variant", sort=False)[[f"acc_{c}" for c in CONDS] + ["heldout_loss", "feat_std_mean", "eff_rank"]].mean()
d = lambda a, b, c: 100 * (m.loc[a, f"acc_{c}"] - m.loc[b, f"acc_{c}"])
print(f"Observed ({STEPS} steps, seeds {SEEDS}; one seed differences of ~1 point are within seed noise):")
for v in VARIANTS:
    print(f"  {v:9s} vs random init: clean {d(v, 'random init (control)', 'clean'):+.1f}, occ40 "
          f"{d(v, 'random init (control)', 'occ40'):+.1f}, noise.15 {d(v, 'random init (control)', 'noise0.15'):+.1f} points | "
          f"held-out loss {m.loc[v, 'heldout_loss']:.3f}, feature std {m.loc[v, 'feat_std_mean']:.3f}, rank {m.loc[v, 'eff_rank']:.1f}")
for v in ("proposed", "ablation"):
    print(f"  {v:9s} vs baseline:    " + ", ".join(f"{c} {d(v, 'baseline', c):+.1f}" for c in CONDS))

**At the assignment budget (`STEPS = 500`)** the pretrained encoders are within about 1 point of the random-encoder
control on clean and occluded images. Only noise σ = 0.15 is higher, by about 5 points. Differences between the
variants are of the size of the seed noise.

Part A of the report found the same over 3 seeds: 35.2 % clean after 500 steps, against 35.1 % for a random encoder. The research question therefore needs the longer budget
the report uses: 15,000 steps on the same splits, seeds and protocol. The next cell rebuilds the report's headline
table from the result files stored in the repository, without retraining.

In [ ]:
# The report's 15k-step results (stored in results/; produced by the same model, data, masks and evaluation protocol)
def proto(path):
    j = json.load(open(path))
    if "rows" in j:                                                  # evaluate.py output
        return [r for r in j["rows"] if r["encoder"] == "pretrained"][0]
    return j["protocol"]["pretrained"]                               # development-notebook run
rr = "results/report_runs"
REPORT = {
    "baseline (5 seeds)": [f"{rr}/h1_masking__baseline/result.json"] + [f"{rr}/h1_masking__baseline__s{k}/result.json" for k in range(1, 5)],
    "proposed: 1_large (5 seeds)": [f"{rr}/seeds_1_large/1_large_pair0/result.json"] + [f"{rr}/seeds_1_large/1_large_pair0_train{k}/result.json" for k in range(1, 5)],
    "ablation: scattered (seed 0)": ["results/h1_perblock/runs/scattered/eval.json"],
    "context noise, H3 (seed 0)": [f"{rr}/noise_aug__ctx_noise/result.json"]}
rep = []
for k, paths in REPORT.items():
    P = [proto(p) for p in paths]
    rep.append({"run (15k steps)": k, **{c: (f"{100*np.mean([p[f'acc_{c}'] for p in P]):.1f}" +
               (f" ± {100*np.std([p[f'acc_{c}'] for p in P], ddof=1):.1f}" if len(P) > 1 else "")) for c in CONDS},
               "feature std": f"{np.mean([p['feat_std_mean'] for p in P]):.3f}", "eff. rank": f"{np.mean([p['eff_rank'] for p in P]):.1f}"})
display(Markdown("**Report results, test accuracy (%), validation-selected probe, 15,000 steps on one L40S**"))
display(pd.DataFrame(rep).set_index("run (15k steps)"))

### 11b. Answers (from the report, 15k steps)

Observed measurements, interpretation and speculation are labelled separately. The report has the full analysis.

**Failure mode (Part A).**
- *Observed:* the baseline does not collapse. Its effective rank grows to about 60/128 and clean accuracy keeps
  rising.
- *Observed:* its accuracy at noise σ = 0.15 falls after about 2k steps, to 22.2 ± 1.0 %. That is below a frozen
  random encoder (26.6 %).
- *Observed:* a probe trained on a mix of clean and noisy images recovers 39.6 % from the same frozen encoder.
- *Interpretation:* the class information survives the noise but moves in feature space, and a clean-only probe
  exposes this.

**Does lower prediction loss correspond to better features?** No (H2 supported).
- *Observed:* the held-out loss is lowest at step 700, when accuracy and rank are still at random-encoder level. It
  then *rises* while clean accuracy improves by 13 points.
- *Observed:* the first drop in the loss is learnt from target positions alone. The loss with another image's context
  falls exactly as fast as with the image's own context.

**Does the change help clean accuracy, occlusion robustness, or both?**
- *Observed (5 seeds each):* `1_large` is the most noise-robust geometry. It reaches 27.5 ± 2.3 % at σ = 0.15,
  against 22.2 ± 1.0 % for the baseline.
- *Observed:* it is +2.3 and +2.8 points at 25 % and 40 % occlusion, and +0.9 points clean.
- *Observed:* it slows the noise failure but does not repair it.

**Geometry or token counts?** Geometry.
- *Observed:* the `scattered` ablation has the same 20 context and 32 target tokens, yet it is the least robust run
  (17.0 % at σ = 0.15).
- *Observed:* it halves the image-specific feature variance, and its held-out loss is close to a prediction from
  target positions alone.
- *Interpretation:* with one large hole, most targets are far from the context and must be inferred from image-level
  content. Small blocks always have context one patch away, so local detail and target positions are enough.
- *Caveat:* contiguity and block scale are confounded.

**Parameters, exposure, probe?** None of them explains the result.
- Every run has 1,412,864 trainable parameters.
- All runs use the same steps, data order and images.
- The probe protocol is identical, with (lr, wd) selected on validation, and the ranking also holds with the fixed
  probe setting.

**Stability from collapse?**
- *Observed:* no run collapses. The effective rank ends at 57–66 and the top-1 singular-value share at ≤ 0.16.
- *Observed:* a frozen random encoder has the most noise-*stable* features (clean–noisy cosine 0.997) and the least
  informative ones. Stability only means something together with rank and accuracy.

**Primary bottleneck.**
- Neither collapse nor context size is the bottleneck: reducing the context or capping target overlap changes
  accuracy by at most 1.5 points.
- Mask geometry decides how much the encoder relies on local detail and positions, and hence how fragile it is to
  noise.
- The main limit is that the objective has no invariance pressure. Adding noise to the context-encoder input during
  pretraining, with targets kept clean, gives 41.9 % at σ = 0.15 with the standard clean probe, for a 1.4-point clean
  cost.

**Next experiment (speculation until run):**
- scale the image and dataset size, and the encoder and predictor;
- measure robustness to noise and occlusion, and to corruptions not seen in training;
- evaluate the features on low-level vision tasks.

**Reproduction.** Set `STEPS = 15000` (and `SEEDS = [0, 1, 2, 3, 4]` for the seed study) and run all cells.
From a shell, the same runs are:
`bash scripts/train.sh configs/<baseline|proposed|ablation>.yaml <seed>` followed by
`bash scripts/evaluate.sh runs/<name>_s<seed>`.